# 🏥 Diagnostic Test Booking System — Data Analysis
### Author: Shubham Jadhav | Data Analyst Portfolio Project

---

**Project Summary:**
End-to-end analysis of **20,000 diagnostic test bookings** across Thane & Navi Mumbai in 2023.
Built on top of a Java-based booking system, this analysis uncovers operational insights
to reduce cancellations, maximize revenue, and improve patient experience.

**Key Focus Areas:**
- Monthly booking trends & revenue growth
- Most popular diagnostic tests
- Cancellation & No-Show root cause analysis
- Center performance comparison
- Home collection demand patterns

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, MySQL


## 1. Setup & Data Loading

In [ ]:
# Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Professional chart styling
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9FAFB',
    'axes.grid':        True,
    'grid.color':       '#E5E7EB',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

BLUE, GREEN, RED, ORANGE, PURPLE, TEAL = '#2563EB','#10B981','#EF4444','#F59E0B','#8B5CF6','#14B8A6'
PALETTE = [BLUE, GREEN, RED, ORANGE, PURPLE, TEAL, '#F97316', '#6366F1', '#84CC16', '#EC4899']
print("Libraries loaded!")

In [ ]:
# Load the dataset
df = pd.read_csv('../data/diagnostic_bookings.csv')

print(f"Total bookings : {len(df):,}")
print(f"Columns        : {df.shape[1]}")
print(f"Date range     : {df['booking_date'].min()} to {df['booking_date'].max()}")
print(f"\nColumn names:")
print(df.columns.tolist())

In [ ]:
df.head()

In [ ]:
print("=== DATA TYPES ===")
print(df.dtypes)
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

In [ ]:
df[['test_price','revenue','report_hours']].describe().round(2)

## 2. Exploratory Data Analysis

In [ ]:
# Booking status breakdown — first thing to check
status = df['status'].value_counts()
print("Booking Status:")
for s, c in status.items():
    print(f"  {s:<12}: {c:>5,}  ({c/len(df)*100:.1f}%)")

print(f"\nCompletion rate : {(df['status']=='Completed').mean()*100:.1f}%")
print(f"Drop-off rate   : {(df['status'].isin(['Cancelled','No-Show'])).mean()*100:.1f}%")

In [ ]:
# Filter completed bookings for revenue analysis
completed = df[df['status'] == 'Completed'].copy()
print(f"Completed bookings : {len(completed):,}")
print(f"Total revenue      : Rs {completed['revenue'].sum()/1e5:.2f} Lakhs")
print(f"Avg booking value  : Rs {df['test_price'].mean():,.2f}")

## 3. Monthly Booking Trends
Are bookings growing? What months drive the most revenue?

In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly = df.groupby('month').agg(
    total_bookings = ('booking_id', 'count'),
    completed      = ('status', lambda x: (x=='Completed').sum()),
    revenue        = ('revenue', 'sum')
).reset_index()

monthly['month_name']    = monthly['month'].apply(lambda x: month_names[x-1])
monthly['completion_pct'] = monthly['completed'] / monthly['total_bookings'] * 100

print(monthly[['month_name','total_bookings','completed','completion_pct','revenue']].to_string(index=False))

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

# Stacked bars: total vs completed
ax1.bar(monthly['month_name'], monthly['total_bookings'], color=BLUE, alpha=0.4, label='Total Bookings')
ax1.bar(monthly['month_name'], monthly['completed'], color=GREEN, alpha=0.85, label='Completed')
ax2.plot(monthly['month_name'], monthly['revenue']/1e5, color=RED, marker='o', linewidth=2.5, label='Revenue (Rs L)')

ax1.set_ylabel('Number of Bookings', fontsize=11)
ax2.set_ylabel('Revenue (Rs Lakhs)', color=RED, fontsize=11)
ax2.tick_params(axis='y', labelcolor=RED)
plt.title('Monthly Booking Volume & Revenue — 2023', fontsize=14, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.show()

print("Insight: Bookings peak in Jan-Mar (winter illness season) and Oct-Nov (annual health checkup season).")

## 4. Diagnostic Test Analysis
Which tests are most popular? Which drive the most revenue?

In [ ]:
test_stats = completed.groupby('test_type').agg(
    bookings = ('booking_id','count'),
    revenue  = ('revenue','sum'),
    avg_price= ('test_price','mean')
).sort_values('bookings', ascending=False)

print("Top 5 Most Booked Tests:")
print(test_stats.head())
print(f"\nTop Revenue Test: {test_stats['revenue'].idxmax()}")
print(f"Avg price (Full Body Checkup): Rs {completed[completed['test_type']=='Full Body Checkup']['test_price'].mean():,.0f}")

In [ ]:
test_plot = test_stats.sort_values('bookings', ascending=True)
colors = [PALETTE[i % len(PALETTE)] for i in range(len(test_plot))]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(test_plot.index, test_plot['bookings'], color=colors, edgecolor='white')
axes[0].set_xlabel('Completed Bookings')
axes[0].set_title('Most Booked Tests', fontsize=12, fontweight='bold')

test_rev = test_stats.sort_values('revenue', ascending=True)
axes[1].barh(test_rev.index, test_rev['revenue']/1e5, color=colors, edgecolor='white')
axes[1].set_xlabel('Revenue (Rs Lakhs)')
axes[1].set_title('Revenue by Test Type', fontsize=12, fontweight='bold')

fig.suptitle('Diagnostic Test Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Insight: CBC is most booked but Full Body Checkup & Thyroid Profile drive higher revenue per booking.")

## 5. Peak Hour Analysis
When do patients book most? Critical for staffing and server capacity.

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_data = df.groupby(['day_of_week','hour']).size().unstack(fill_value=0).reindex(day_order)

peak_hour = df['hour'].value_counts().idxmax()
peak_day  = df['day_of_week'].value_counts().idxmax()
print(f"Peak booking hour : {peak_hour}:00")
print(f"Busiest day       : {peak_day}")
print(f"\nHourly breakdown (top 5):")
print(df['hour'].value_counts().head())

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(heatmap_data, cmap='Blues', ax=ax, linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Number of Bookings'})
ax.set_xlabel('Hour of Day (0=Midnight, 12=Noon)')
ax.set_ylabel('Day of Week')
ax.set_title('Booking Volume Heatmap — Peak Hours by Day', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Insight: Evening hours (6PM-9PM) are peak booking time — patients book after work.")
print("Recommendation: Ensure app/website can handle load during 6-9PM window.")

## 6. Cancellation & No-Show Analysis
Reducing drop-offs is directly linked to revenue recovery.

In [ ]:
# Cancellation by channel
cancel_channel = df.groupby('booking_channel').apply(
    lambda x: (x['status'].isin(['Cancelled','No-Show'])).mean() * 100
).sort_values(ascending=False).reset_index()
cancel_channel.columns = ['channel','dropoff_rate']

print("Drop-off Rate by Booking Channel:")
print(cancel_channel.to_string(index=False))

# Revenue lost to cancellations
cancelled = df[df['status'].isin(['Cancelled','No-Show'])]
lost_revenue = cancelled['test_price'].sum()
print(f"\nEstimated Revenue Lost to Drop-offs: Rs {lost_revenue/1e5:.2f} Lakhs")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Status pie
status_counts = df['status'].value_counts()
axes[0].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
            colors=[GREEN, BLUE, RED, ORANGE], startangle=140,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Booking Status Breakdown', fontweight='bold')

# Cancellation by channel
axes[1].bar(cancel_channel['channel'], cancel_channel['dropoff_rate'],
            color=RED, alpha=0.8, edgecolor='white')
axes[1].set_ylabel('Drop-off Rate (%)')
axes[1].set_title('Drop-off Rate by Channel', fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(cancel_channel['dropoff_rate']):
    axes[1].text(i, v+0.1, f'{v:.1f}%', ha='center', fontsize=9)

# Cancellation by age group
age_order = ['18-25','26-35','36-45','46-55','56-65','65+']
cancel_age = df.groupby('age_group').apply(
    lambda x: (x['status'].isin(['Cancelled','No-Show'])).mean() * 100
).reindex(age_order)
axes[2].bar(cancel_age.index, cancel_age.values, color=ORANGE, alpha=0.85, edgecolor='white')
axes[2].set_ylabel('Drop-off Rate (%)')
axes[2].set_title('Drop-off Rate by Age Group', fontweight='bold')

fig.suptitle('Cancellation & No-Show Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Insight: Walk-in bookings have highest drop-off. SMS reminders for older age groups could reduce no-shows.")

## 7. Revenue & Center Performance

In [ ]:
# Center performance
center_perf = df.groupby('diagnostic_center').agg(
    total     = ('booking_id','count'),
    completed = ('status', lambda x: (x=='Completed').sum()),
    revenue   = ('revenue','sum'),
    avg_price = ('test_price','mean')
).reset_index()
center_perf['completion_rate'] = center_perf['completed']/center_perf['total']*100
center_perf['short']           = center_perf['diagnostic_center'].apply(lambda x: x.split(' - ')[0])
center_perf_sorted             = center_perf.sort_values('revenue', ascending=False)
print(center_perf_sorted[['short','total','completed','completion_rate','revenue']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue by center
cp = center_perf.sort_values('revenue', ascending=True)
axes[0].barh(cp['short'], cp['revenue']/1e5, color=PALETTE[:len(cp)], edgecolor='white')
axes[0].set_xlabel('Revenue (Rs Lakhs)')
axes[0].set_title('Revenue by Diagnostic Center', fontweight='bold')
for i, v in enumerate(cp['revenue']/1e5):
    axes[0].text(v+0.1, i, f'Rs {v:.1f}L', va='center', fontsize=9)

# Revenue by channel
channel_rev = completed.groupby('booking_channel')['revenue'].sum().sort_values(ascending=False)
axes[1].bar(channel_rev.index, channel_rev.values/1e5, color=PALETTE[:len(channel_rev)], edgecolor='white')
axes[1].set_ylabel('Revenue (Rs Lakhs)')
axes[1].set_title('Revenue by Booking Channel', fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

fig.suptitle('Revenue Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Insight: Mobile App drives highest revenue. Investing in app UX improvements would have highest ROI.")

## 8. Home Collection Analysis

In [ ]:
home_stats = df.groupby('city').agg(
    total=('booking_id','count'),
    home_collection=('home_collection','sum')
).reset_index()
home_stats['home_pct'] = home_stats['home_collection']/home_stats['total']*100
home_stats = home_stats.sort_values('home_pct', ascending=False)

print("Home Collection Demand by City:")
print(home_stats[['city','total','home_collection','home_pct']].to_string(index=False))
print(f"\nOverall home collection rate: {df['home_collection'].mean()*100:.1f}%")

## 9. Key Findings & Business Recommendations

---

### Summary

| Metric | Value |
|--------|-------|
| Total Bookings | 20,000 |
| Completion Rate | 72.0% |
| Cancellation Rate | 8.1% |
| No-Show Rate | 5.1% |
| Total Revenue | Rs 84.25 Lakhs |
| Avg Booking Value | Rs 584 |
| Home Collection Rate | 25.0% |
| Most Booked Test | Complete Blood Count (CBC) |
| Top Channel | Mobile App (35%) |
| Peak Booking Hour | 7:00 PM |

---

### Business Recommendations

1. **Reduce No-Shows (5.1% → target 2%)** — Automated SMS/WhatsApp reminders 24hrs and 2hrs before appointment could recover ~Rs 8-10 Lakhs annually.

2. **Channel Strategy** — Mobile App drives the highest revenue and lowest drop-off rate. Investing in app features (easy reschedule, test reminders) will have highest ROI.

3. **Evening Staffing** — Peak bookings at 7PM–9PM. Ensure customer support and home collection staff are available in this window.

4. **High-Value Test Promotion** — Full Body Checkup and Thyroid Profile have the highest revenue per booking. Bundling these with preventive health packages could increase average ticket size.

5. **Home Collection Expansion** — 25% of patients prefer home collection. Expanding fleet in high-demand cities (Thane, Navi Mumbai) would capture more bookings.

---

*Project built on top of a Java-based Diagnostic Booking System. Dataset is synthetically generated to mirror real-world booking patterns.*
